# A12. Patent/Trademark Risk Quick Screening Notebook

> **Related Module**: [A12 Intellectual Property Protection](../paths/a-operators/a12-ip-protection.md)
>
> **Features**: Input product information → Search Google Patents → Risk assessment + Recommendations
>
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kangise/ecommerce-ai-roadmap/blob/main/notebooks/a12-ip-patent-search.ipynb)

In [ ]:
!pip install -q requests pandas openai

## 1. Input Product Information

In [ ]:
import requests
import pandas as pd
import json
import os
from openai import OpenAI

# === Replace with your product ===
PRODUCT = {
    'name': 'Portable Neck Fan',
    'category': 'Personal Electronics',
    'key_features': ['bladeless design', 'LED display', '3 speed settings', 'USB-C charging', 'wearable'],
    'target_market': 'US',
    'visual_features': 'Arc-shaped band worn around neck, no visible blades, small LED screen on front'
}

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', 'your-key-here')
client = OpenAI(api_key=OPENAI_API_KEY)

print(f'Product: {PRODUCT["name"]}')
print(f'Features: {", ".join(PRODUCT["key_features"])}')
print(f'Market: {PRODUCT["target_market"]}')

## 2. Google Patents Search

In [ ]:
# Google Patents search (via SerpAPI or by constructing URLs directly)
# Note: This generates search URLs; actual searching requires manual action or an API

search_queries = []
for feature in PRODUCT['key_features']:
    q = f'{PRODUCT["name"]} {feature}'
    search_queries.append(q)

# Additional patent search strategies
search_queries.extend([
    f'{PRODUCT["category"]} {PRODUCT["key_features"][0]}',
    f'wearable fan bladeless patent',
    f'neck fan LED display utility patent',
])

print('=== Patent Search Queries ===')
print('Copy these to https://patents.google.com/ to search:\n')
for i, q in enumerate(search_queries, 1):
    url = f'https://patents.google.com/?q={q.replace(" ", "+")}&country=US'
    print(f'{i}. "{q}"')
    print(f'   {url}\n')

print(f'\nAlso check:')
print(f'  USPTO: https://ppubs.uspto.gov/pubwebapp/')
print(f'  Lens.org: https://www.lens.org/lens/search/patent/list?q={PRODUCT["name"].replace(" ", "+")}')

## 3. AI Risk Assessment

In [ ]:
prompt = f"""You are an intellectual property risk assessment expert for e-commerce products.

Product: {PRODUCT['name']}
Category: {PRODUCT['category']}
Key Features: {', '.join(PRODUCT['key_features'])}
Visual Design: {PRODUCT['visual_features']}
Target Market: {PRODUCT['target_market']}

Please assess IP risks and return JSON:
{{
  "overall_risk": "HIGH/MEDIUM/LOW",
  "patent_risk": {{
    "level": "HIGH/MEDIUM/LOW",
    "reason": "explanation",
    "high_risk_features": ["features most likely patented"],
    "search_keywords": ["recommended patent search terms"]
  }},
  "design_patent_risk": {{
    "level": "HIGH/MEDIUM/LOW",
    "reason": "explanation"
  }},
  "trademark_risk": {{
    "level": "HIGH/MEDIUM/LOW",
    "reason": "explanation",
    "names_to_avoid": ["brand names that might conflict"]
  }},
  "tro_risk": {{
    "level": "HIGH/MEDIUM/LOW",
    "reason": "has this category had frequent TRO cases?"
  }},
  "recommendation": "GO / CAUTION / NO-GO",
  "next_steps": ["ordered list of actions before selling"]
}}"""

response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': prompt}],
    response_format={'type': 'json_object'},
    max_tokens=1000
)

assessment = json.loads(response.choices[0].message.content)

risk_emoji = {'HIGH': '🔴', 'MEDIUM': '🟡', 'LOW': '🟢'}
print(f'=== IP Risk Assessment: {PRODUCT["name"]} ===')
print(f'\nOverall Risk: {risk_emoji.get(assessment["overall_risk"], "")} {assessment["overall_risk"]}')
print(f'Recommendation: {assessment["recommendation"]}')

for risk_type in ['patent_risk', 'design_patent_risk', 'trademark_risk', 'tro_risk']:
    r = assessment.get(risk_type, {})
    level = r.get('level', 'N/A')
    print(f'\n{risk_emoji.get(level, "")} {risk_type.replace("_", " ").title()}: {level}')
    print(f'   {r.get("reason", "")}')

print(f'\n=== Next Steps ===')
for i, step in enumerate(assessment.get('next_steps', []), 1):
    print(f'{i}. {step}')

## 4. Export

In [ ]:
with open('ip_risk_assessment.json', 'w') as f:
    json.dump(assessment, f, indent=2)
print('Exported to ip_risk_assessment.json')
print('\n⚠️ AI assessment is for initial screening only. Consult a patent attorney for high-risk products.')